# 正解 — Stretch 03 — 状態空間モデル

Species_short × yearの平均Body_Massパネルで、種ごとのローカルレベル状態空間をPyMCで推定してください。あわせて`pm.model_to_graphviz(model)`でモデル図をノートブックに表示し、種別levelの事後平均・区間と観測平均を重ねた図をノートブックに表示してください。

種混合の年次平均は使わず、種別系列にしてください。

`python/` に入って実行してください（データは `../../../data/`）。


## 準備


In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
from IPython.display import display

SEED = 123

df = pd.read_parquet("../../../data/penguins.parquet")


## パネルデータ

Species_short × year の平均Body_Massパネルを作り、species_idx / time_idx / coords を用意してください。

`coords["time"]` の年ラベルは Python の `int` にしてください（`Date_Egg.dt.year` 由来の `np.int32` のままだと nutpie が `Coordinate time value has unsupported type` で落ちます）。


In [ ]:
df = df.dropna(subset=["Date_Egg", "Body_Mass", "Species_short"])
panel = (
    df.assign(year=lambda d: d["Date_Egg"].dt.year)
    .groupby(["Species_short", "year"], observed=True)["Body_Mass"]
    .mean()
    .reset_index()
    .sort_values(["Species_short", "year"])
)
panel["species_idx"], species_levels = pd.factorize(panel["Species_short"].astype(str))
# nutpie rejects np.int32 coord labels; use plain Python ints
years = sorted(int(y) for y in panel["year"].unique())
panel["time_idx"] = panel["year"].map({y: i for i, y in enumerate(years)})
n_species = len(species_levels)
T = len(years)
y = panel["Body_Mass"].to_numpy()

coords = {
    "species": list(species_levels),
    "time": years,
    "obs": np.arange(len(panel)),
}


## モデル・graphviz・サンプリング

種ごとのローカルレベル状態空間を定義し、`model_to_graphviz`をノートブックに表示してからサンプリングしてください（`random_seed=123`）。


In [ ]:
with pm.Model(coords=coords) as model:
    sigma_level = pm.HalfNormal("sigma_level", 100)
    sigma_obs = pm.HalfNormal("sigma_obs", 200)
    # 種ごとの初期水準（固定効果に近い役割）
    mu0 = pm.Normal("mu0", 4000, 500, dims="species")

    innov = pm.Normal("innov", 0, 1, dims=("species", "time"))
    level = pm.Deterministic(
        "level",
        mu0[:, None] + pm.math.cumsum(innov, axis=1) * sigma_level,
        dims=("species", "time"),
    )
    mu_obs = level[panel["species_idx"].to_numpy(), panel["time_idx"].to_numpy()]
    pm.Normal("y", mu=mu_obs, sigma=sigma_obs, observed=y, dims="obs")

    graph = pm.model_to_graphviz(model)
    display(graph)

    idata = pm.sample(
        draws=400,
        tune=400,
        chains=4,
        target_accept=0.95,
        random_seed=SEED,
        progressbar=True,
    )


## 要約と診断

`sigma_level` / `sigma_obs` / `mu0` の要約とR-hatを表示してください。


In [ ]:
summary = az.summary(idata, var_names=["sigma_level", "sigma_obs", "mu0"], round_to=2)
print(f"{summary=}")
max_rhat = float(summary["r_hat"].max())
if max_rhat > 1.05:
    print(
        f"NOTE: {max_rhat=:.3f} (>1.05). "
        "デモ設定のため不安定なことがあります。講師デモを参照して構いません。"
    )
else:
    print(f"R-hat OK ({max_rhat=:.3f})")
print(f"{list(species_levels)=}, {years=}, {len(panel)=}, {n_species=}, {T=}")


## level図

種別levelの事後平均・94%区間と観測平均を重ねた図をノートブックに表示してください。


In [ ]:
# 推定結果の可視化: 種別 level の事後平均・94% 区間と観測平均
level_da = idata.posterior["level"]  # chain, draw, species, time
level_mean = level_da.mean(dim=("chain", "draw"))
level_lo = level_da.quantile(0.03, dim=("chain", "draw"))
level_hi = level_da.quantile(0.97, dim=("chain", "draw"))
year_x = np.asarray(years, dtype=float)

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = ["#0072B2", "#E69F00", "#009E73"]  # Okabe–Ito 系
for s_i, sp in enumerate(species_levels):
    means = np.asarray(level_mean.sel(species=sp).values, dtype=float)
    lo = np.asarray(level_lo.sel(species=sp).values, dtype=float)
    hi = np.asarray(level_hi.sel(species=sp).values, dtype=float)
    c = colors[s_i % len(colors)]
    ax.fill_between(year_x, lo, hi, color=c, alpha=0.2)
    ax.plot(year_x, means, color=c, marker="o", label=f"{sp} level")
    obs = panel.loc[panel["Species_short"] == sp].sort_values("year")
    ax.scatter(
        obs["year"].to_numpy(dtype=float),
        obs["Body_Mass"].to_numpy(dtype=float),
        color=c,
        marker="x",
        s=60,
        zorder=3,
    )
ax.set_xlabel("year")
ax.set_ylabel("Body_Mass (g)")
ax.set_title("State-space levels (mean + 94% interval) vs observed yearly means")
ax.legend(frameon=False, loc="best")
fig.tight_layout()
plt.show()
